# noise_reachability step 2: noise selection per chunk (Colab, A100)

Runs `noise_reachability/run_step2_noise_selection.sh`: the frozen `lerobot/smolvla_libero` at `n_action_steps=10`
and policies that draw N noise samples per inference and execute the chunk a score picks, paired episode by episode
on tasks 0, 4, 6, 7, 8. The protocol, hypothesis and selection rule are in `noise_reachability/README.md` (step 2).

**Two stages, two sessions** (`STAGE` in step 1):

1. `STAGE = "select"`: init states 10-19 x seeds 0-1, six policies (600 episodes). Step 9 applies the selection
   rule and prints the chosen policy, or that step 2 stops.
2. `STAGE = "report"` with `REPORT_POLICY` set to the chosen policy: init states 0-9 x seeds 0-2, base and the
   chosen policy (300 episodes). Step 9 prints the test of the hypothesis.

**Before you start**

Runtime > Change runtime type > **A100 GPU**, and **High-RAM** if offered: each task runs in its own process
(about 5 GB of RAM each), and step 2 sets how many run at once from the RAM available.

**How to run.** Run the cells in order. Step 7 starts the run in the background; step 8 keeps a cell busy until it
ends, so Colab does not treat the runtime as idle. Keep the tab open and the computer awake.

**If the runtime is lost:** results are written to Google Drive episode by episode. Run steps 1 to 7 again; finished
episodes are skipped.

Keep every episode of this comparison on Colab; do not pool them with local or Kaggle episodes.

## 1. Settings

In [ ]:
import csv, glob, json, os, re, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "main"
REPO = "/content/efficient_vla"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"

STAGE = "select"          # "select" first; then "report" with REPORT_POLICY from the select stage's step 9
REPORT_POLICY = ""        # e.g. "medoid_n8"; used only when STAGE = "report"
TASKS = "0 4 6 7 8"

DRIVE_DIR = "/content/drive/MyDrive/efficient_vla_runs/noise_selection"
OUT_ROOT = f"{DRIVE_DIR}/{STAGE}"   # results and logs; the run resumes from them
SELECT_POLICIES = "base,first_n8,medoid_n8,accel_n8,medoid_n16,accel_n16"
POLICIES = SELECT_POLICIES if STAGE == "select" else f"base,{REPORT_POLICY}"
assert STAGE in ("select", "report") and (STAGE == "select" or REPORT_POLICY), "set REPORT_POLICY for the report stage"

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
print(f"stage {STAGE}, policies {POLICIES}, tasks {TASKS}, output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r

## 2. GPU, RAM and Drive

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
assert smi.stdout.strip(), "No GPU. Set the runtime type to A100 GPU."
if "A100" not in smi.stdout:
    print("WARNING: not an A100; the run works but is slower")
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
PARALLEL = max(1, min(len(TASKS.split()), int((ram_gb - 4) // 5)))
print(f"driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; {ram_gb:.0f} GB RAM -> {PARALLEL} tasks at once")

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT_ROOT, exist_ok=True)

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

A separate virtual environment with the package versions of the local `lerobot_env` (the same cell as
`continued_training/colab_continued_training.ipynb`). Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks and tests

PyTorch sees the GPU and runs bfloat16, MuJoCo renders headless, and the repo's CPU tests pass (including
`tests/test_noise_selection.py`).

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", PYTHONPATH=REPO, PYTHON_BIN=PY)

sh(f"""{PY} -c "
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
x = torch.randn(256, 256, device='cuda', dtype=torch.bfloat16)
print('bfloat16 matmul ok', float((x @ x).float().abs().mean()) > 0)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)

## 6. Smoke check (one episode, about 3 minutes)

One `medoid_n8` episode of task 0 in a fresh scratch folder: the model downloads and the selector runs. The log line
shows the outcome, the number of inferences (52 if the episode runs all 520 steps, fewer if it succeeds earlier)
and how often each of the 8 samples was chosen. Either outcome is fine; not part of the results.

In [ ]:
smoke = "/content/step2_smoke"
shutil.rmtree(smoke, ignore_errors=True)   # a re-run of this cell runs the episode again
r = sh(f"bash {REPO}/noise_reachability/run_step2_noise_selection.sh", check=False,
       env=dict(RUN_ENV, SMOKE="1", POLICIES="medoid_n8", OUT_ROOT=smoke))
rows = list(csv.DictReader(open(f"{smoke}/task_0/per_episode_medoid_n8.csv"))) if r.returncode == 0 else []
ok = len(rows) == 1 and re.search(r"inferences=\d+ chosen_per_sample=", r.stdout)
assert ok, "smoke episode did not finish; see the output above"
print("smoke episode ok:", rows[0])

## 7. Launch (background)

One runner per task, `PARALLEL` at a time. Running this cell while a run is active does nothing; after a lost
runtime it resumes.

In [ ]:
running = subprocess.run(["pgrep", "-f", "run_step2_noise_[s]election"], capture_output=True, text=True).stdout.split()
if running:
    print("A run is already active (pids:", " ".join(running) + "). Not starting another.")
else:
    log = f"{OUT_ROOT}/runner.log"
    cmd = (f"printf '%s\\n' {TASKS} | xargs -P {PARALLEL} -I{{}} env TASKS={{}} "
           f"bash {REPO}/noise_reachability/run_step2_noise_selection.sh > /dev/null")
    subprocess.Popen(f"nohup bash -c \"{cmd}; echo runner exit \\$?\" >> '{log}' 2>&1 &", shell=True,
                     executable="/bin/bash", env=dict(RUN_ENV, STAGE=STAGE, POLICIES=POLICIES, OUT_ROOT=OUT_ROOT),
                     start_new_session=True)
    sh(f"git -C {REPO} log -1 --format='%H' > '{OUT_ROOT}/repo_commit.txt'")
    print("Started. Per-task logs:", f"{OUT_ROOT}/task_<t>/eval.log")

## 8. Progress; waits until the run ends

Prints episodes done per policy every 10 minutes while a runner is active. Interrupting this cell does not stop
the run; re-run it any time (needs only steps 1 and 2).

In [ ]:
def progress():
    total = 0
    for t in TASKS.split():
        counts = []
        for p in POLICIES.split(","):
            f = Path(OUT_ROOT, f"task_{t}", f"per_episode_{p}.csv")
            rows = list(csv.DictReader(open(f))) if f.exists() else []
            total += len(rows)
            counts.append(f"{p} {len(rows)} ({sum(int(r['success']) for r in rows)} ok)")
        print(f"task {t}: " + ", ".join(counts))
    return total

while True:
    print(time.strftime("%H:%M"), flush=True)
    done = progress()
    if not subprocess.run(["pgrep", "-f", "run_step2_noise_[s]election"], capture_output=True).stdout:
        break
    time.sleep(600)
print("no runner active;", done, "episodes recorded")
sh(f"tail -n 3 '{OUT_ROOT}/runner.log'", check=False)

## 9. Results

Standard library only (`python3 -S`): works on a fresh runtime after steps 1 to 3. Fails if an episode is missing.

In [ ]:
tasks = " ".join(TASKS.split())
if STAGE == "select":
    sh(f"python3 -S {REPO}/noise_reachability/analyze_step2.py --stage select --root '{OUT_ROOT}' --tasks {tasks} "
       f"--policies {POLICIES} --out '{OUT_ROOT}/step2_select.txt'")
else:
    sh(f"python3 -S {REPO}/noise_reachability/analyze_step2.py --stage report --root '{OUT_ROOT}' --tasks {tasks} "
       f"--policies {POLICIES} --step1 {REPO}/week6/noise_reachability_step1_kaggle/step1_reachability.txt.json "
       f"--lerobot_eval {REPO}/week6/noise_reachability_step0_kaggle/nas10_all_tasks/per_episode.csv "
       f"--out '{OUT_ROOT}/step2_report.txt'")

## 10. Download the results for the repo

Per-episode files, logs, provenance and the step-9 report.

In [ ]:
name = f"noise_selection_{STAGE}_colab"
sh(f"cd '{OUT_ROOT}' && zip -q -r /content/{name}.zip . -x '*.pyc'")
sh(f"du -h /content/{name}.zip")
from google.colab import files
files.download(f"/content/{name}.zip")